In [1]:
# Import libraries
import numpy as np
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
import urllib3 as ul
import requests
import json
from pathlib import Path
import os

##### References cited while developing code for data fetching

y_finance download: https://stackoverflow.com/questions/63107594/how-to-deal-with-multi-level-column-names-downloaded-with-yfinance/63107801#63107801

In [ ]:
data = yf.Ticker("MSFT")
print(data)

In [ ]:
history = data.history(period = '1y')

# Dividends and Stock Splits are not required for our current analysis
history.drop(['Dividends', 'Stock Splits'], axis=1, inplace= True)

In [ ]:
history.info()

In [ ]:
history.ffill()
print(history)

In [ ]:
# Defining pool of assets
"""
DataPool
Responsibiliy:
1: Define the asset pool for which we need a data
    - specifies ticker or identifiers for each asset under analysis
"""

# Fetching news data

"""
Data Fetcher
Responsibility: 
1. Fetch data from various sources 2: Alpaca, Yahoo Finance
    - fetch in batches to get maximum data considering the rate limit from alpaca
2. Store the data locally in a structured format (Bronze layer)
    - store it in json files locally, append only

Things to take care of:
1. Contains and maintains the raw state of the data source in its original formats. Serves as the single source of truth.
2. Immutability: The data fetched should be immutable. Once the data is fetched, it should be stored in the data lake and should not be modified.
3. Error handling: If the data is not fetched correctly, it should be logged in the data lake.
4. Consider rate limits for each API. - Round Robin
5. Tag every ingest record with ingest_id, provider, and fetch_timestamp.
"""


In [2]:
# API reference: https://docs.alpaca.markets/reference/news-3
url = "https://data.alpaca.markets/v1beta1/news?" # Alpaca market data api endpoint for news data

"""
required inputs;
start: start date for the news data
end: end date for the news data
sort: sort the news data by the date of the news ['asc', 'desc']
symbols: list of symbols for which the news data is to be fetched
limit: max 50, default 10
include_content: boolean to include the content of the news (Always true)
exclude_contentless: boolean to exclude the news with no content (Always True - prefer with content)
page_token: will be decided letter - irrelevant for now
"""
headers = {
    "accept": "application/json",
    "APCA-API-KEY-ID": "PKDOOOL44COZGZB6E7KNQAFMOF",
    "APCA-API-SECRET-KEY": "FZSHhs41VmSSU8z7otcQfHMJEFFuqHzSuMLZVuixrexZ"
}
symbols = ['MSFT', 'AMZN']
query_params = {
    "start": '2024-01-03T00:00:00Z', #Update later based on the continous fetching logic
    "end": '2025-10-03T00:00:00Z', #Update later based on the continous fetching logic
    "sort": 'desc', #Always ascending - latest articls at last
    "symbols": ','.join(symbols), # Aligned with expectations of Alpaca ',' string #Update later based on the continous fetching logic
    "limit": 50,
    "include_content": True,
    "exclude_contentless": True,
    "page_token": '' #extract from the responses if more articles left to fetch
    }

# response = requests.get(url, params=query_params, headers=headers)

# print(type(response))
# print(response)
# print(response.text)

In [ ]:
print(query_params.items())

In [3]:
# Evrything related to path

# Why Pathlib? 
# 1. avoid traditional string paths 2. reuqires only one module to be imported
# 3. Object-oriented interface to paths - allows direct system calls rather than just string manipulation

# Instantiate Path object with the projects directory to access all the paths within the project
home_path = Path(Path.cwd().parent)
path_to_raw_data = home_path/'data/dump/alpaca'

In [ ]:
# Store data in a raw format locally

#steps
# future: conditional path for differe|nt data sources
#1. Serialize json response
#2. Open the file in that path using with keyword
#3. Append text in that file

with open(f'{path_to_raw_data}/data.json', 'a') as f:
    json.dump(response.text, f) #need to review because, if already parsed earlier, no need to parse it again

In [8]:
# API Call functions

# rate limit - 200 calls/min
# time limit - data is only available since 2017

def fetch_data(url, query_params, headers):
    response = requests.get(url, params=query_params, headers=headers)
    response_dict = (json.loads(response.text))
    news_text = response_dict['news']
    if 'next_page_token' in response_dict:
        next_page_token = response_dict['next_page_token']
        query_params['page_token'] = next_page_token
        while(next_page_token):
            response = requests.get(url, params=query_params, headers=headers)
            response_dict = (json.loads(response.text))
            news_text.append(str(response_dict['news'])) 
            next_page_token = response_dict['next_page_token']
            query_params['page_token'] = next_page_token
    else:
        print('Fetched all the data')
    return news_text

text_to_write = fetch_data(url, query_params, headers)
  
# features required
# 1. Make calls to get all the news starting from 2017 for each symbol in a way that considers rate limit

# steps extract next_page_token from response

 #load - can only be used with file object

In [13]:
with open(f'{path_to_raw_data}/data.json', 'w') as f:
    json.dump(text_to_write, f)